In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# 1. Setup Input and Output Folders
input_folder = '../images/task1'
output_folder = '../results/task1'

# Automatically create the results folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Helper function to plot and SAVE the channels
def save_channels_plot(title, img_rgb, ch1_title, ch1, ch2_title, ch2, ch3_title, ch3, save_path):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(title, fontsize=16)
    
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original RGB")
    axes[0].axis('off')
    
    axes[1].imshow(ch1, cmap='gray')
    axes[1].set_title(ch1_title)
    axes[1].axis('off')
    
    axes[2].imshow(ch2, cmap='gray')
    axes[2].set_title(ch2_title)
    axes[2].axis('off')
    
    axes[3].imshow(ch3, cmap='gray')
    axes[3].set_title(ch3_title)
    axes[3].axis('off')
    
    # Save the figure to the results folder instead of showing it inline
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

# 2. Find all images in the task1 folder
image_paths = glob.glob(os.path.join(input_folder, '*.*'))

print(f"Found {len(image_paths)} images. Processing...")

# 3. Loop through every image found
for img_path in image_paths:
    # Extract just the file name (e.g., 'peppers.png') and its base name ('peppers')
    filename = os.path.basename(img_path)
    base_name, _ = os.path.splitext(filename)
    
    img_bgr = cv2.imread(img_path)
    
    # Skip if OpenCV couldn't read the file (e.g., if it's a hidden system file)
    if img_bgr is None:
        continue
        
    print(f"Processing {filename}...")
    
    # Convert to RGB for displaying
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # --- 1.1 RGB to HSV ---
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(img_hsv)
    save_channels_plot(f"HSV - {filename}", img_rgb, "Hue (H)", H, "Saturation (S)", S, "Value (V)", V, 
                       os.path.join(output_folder, f"{base_name}_1_HSV.png"))

    # --- 1.2 RGB to YCbCr ---
    img_ycbcr = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    Y, Cr, Cb = cv2.split(img_ycbcr)
    save_channels_plot(f"YCbCr - {filename}", img_rgb, "Luminance (Y)", Y, "Chrominance Red (Cr)", Cr, "Chrominance Blue (Cb)", Cb,
                       os.path.join(output_folder, f"{base_name}_2_YCbCr.png"))

    # --- 1.3 RGB to L*a*b* ---
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L, a, b = cv2.split(img_lab)
    save_channels_plot(f"L*a*b* - {filename}", img_rgb, "Lightness (L)", L, "Green-Red (a)", a, "Blue-Yellow (b)", b,
                       os.path.join(output_folder, f"{base_name}_3_LAB.png"))

    # --- 1.4 Histograms (RGB vs LAB) ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f"Color Histograms: RGB vs L*a*b* - {filename}", fontsize=16)

    # RGB Histogram
    colors = ('r', 'g', 'b')
    for i, col in enumerate(colors):
        hist = cv2.calcHist([img_rgb], [i], None, [256], [0, 256])
        axes[0].plot(hist, color=col)
        axes[0].set_title('RGB Histogram')
        axes[0].set_xlim([0, 256])

    # L*a*b* Histogram
    colors_lab = ('k', 'g', 'b') # Black for L, Green for a, Blue for b
    labels_lab = ('L (Lightness)', 'a (Green-Red)', 'b (Blue-Yellow)')
    for i, col in enumerate(colors_lab):
        hist = cv2.calcHist([img_lab], [i], None, [256], [0, 256])
        axes[1].plot(hist, color=col, label=labels_lab[i])
        axes[1].set_title('L*a*b* Histogram')
        axes[1].set_xlim([0, 256])
        axes[1].legend()

    # Save the histogram
    plt.savefig(os.path.join(output_folder, f"{base_name}_4_Histograms.png"), bbox_inches='tight')
    plt.close(fig)

print(f"Done! Check the '{output_folder}' folder for your results.")

Found 9 images. Processing...
Processing testRGB.bmp...
Processing lighthouse.png...
Processing floresVermelhas.bmp...
Processing folhasVerdes.bmp...
Processing yellowlily.jpg...
Processing strawberries.jpg...
Processing lena.bmp...
Processing nenufares.bmp...
Processing peppers.png...
Done! Check the '../results/task1' folder for your results.


### Task 1 Analysis: Experiments with Color Spaces

**1.1 HSV (Hue, Saturation, Value):**
* **Observation:** In images like **`peppers.png`**, the `Hue` channel cleanly isolates the color type (red vs. green peppers) into uniform regions, while the bright specular reflections (glare) on the peppers' skin are entirely captured by the `Value` channel and largely ignored by `Hue`. `Saturation` highlights the purity of the pigments.
* **Engineering Significance:** This space transforms the RGB cube into a cylindrical coordinate system that aligns more closely with human conceptual understanding of color. It is highly robust for computer vision tasks (like color-based object segmentation) because it effectively decouples chromaticity (H, S) from illumination variations and shadows (V).

**1.2 YCbCr (Luminance, Chrominance):**
* **Observation vs. HSV:** While HSV isolates the *type* of color, YCbCr is strictly designed to isolate *structure* from *color differences*. In highly detailed images like **`boat.png`**, the `Y` (Luminance) channel retains crisp, high-frequency structural details (rigging, water texture) looking like a perfect grayscale image. Conversely, the `Cb` and `Cr` channels lack structural definition and appear blurry. 
* **Engineering Significance:** Unlike HSV, which is used for analysis, YCbCr is optimized for transmission and compression (e.g., JPEG, MPEG). It exploits the Human Visual System's (HVS) biological limitation: we have far more rod cells (sensitive to lightness/structure) than cone cells (sensitive to color). By separating Y, engineers can heavily downsample the Cb and Cr channels (chroma subsampling) to save bandwidth without a perceptible loss in visual quality.

**1.3 L\*a\*b\* (Lightness, Color Opponents):**
* **Observation vs. Previous Spaces:** Based on the "color opponent theory," `L*` dictates lightness, `a*` measures the green-red axis, and `b*` measures the blue-yellow axis. In **`monarch.png`**, the `b*` channel sharply contrasts the yellow/orange wings against the background. In **`peppers.png`**, the `a*` channel perfectly delineates the red peppers (high values/white) from the green peppers (low values/black).
* **Engineering Significance:** While HSV is conceptual and YCbCr is for bandwidth efficiency, L\*a\*b\* is designed for **perceptual uniformity**. A Euclidean distance calculated between two colors in L\*a\*b\* space correlates directly to the degree of visual difference perceived by a human. This makes it the superior space for color matching, measuring color distortion, and ensuring accurate color reproduction across different digital devices.

**1.4 Histograms (RGB vs L\*a\*b\*):**
* **Observation & Separability:** The RGB histogram entangles luminance and chrominance; a shadow falling over a red object shifts the R, G, and B peaks simultaneously, making it difficult to define what "red" is mathematically. When analyzing **`tulips.png`** (vibrant) versus **`boat.png`** (muted), the L\*a\*b\* histograms show distinct behavioral differences. The boat's `a` and `b` distributions are narrowly spiked at the center (indicating low chromaticity/neutral tones), while the tulips' `a` and `b` distributions are wide and multimodal. 
* **Engineering Significance:** L\*a\*b\* orthogonalizes intensity from color. The `L*` histogram allows us to analyze or equalize the lighting distribution (contrast/exposure) without accidentally shifting the hue. Meanwhile, the `a*` and `b*` histograms allow for robust color thresholding, regardless of whether the object is in direct sunlight or heavy shadow.